# Report Analytics

## Report-Level Behavioural Analytics

This notebook adds a simple, explainable analytics layer on top of the processed Power BI usage data. The goal is to compute report-level behavioural features, group reports into business-friendly segments, and apply diagnostic rules that help identify health risks.

The reusable implementation lives in `src/analytics/`. This notebook is the portfolio-friendly walkthrough layer.

## 1. Project Context

The forecasting pipeline estimates future report usage. This analytics layer is separate: it explains current report behaviour using processed semantic model tables, without changing forecasting logic or regenerating synthetic source data.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.analytics.report_features import build_report_features
from src.analytics.report_segmentation import build_report_segments
from src.analytics.report_diagnostics import build_report_diagnostics

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
SEGMENTS_DIR = PROJECT_ROOT / "outputs" / "segments"
DIAGNOSTICS_DIR = PROJECT_ROOT / "outputs" / "diagnostics"

for output_dir in [METRICS_DIR, SEGMENTS_DIR, DIAGNOSTICS_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Processed Input Tables

The workflow uses the best available processed tables. `mart_forecast_features.csv` is the richest daily usage mart, while the report views fact table supplies user-level repeat and concentration measures.

In [2]:
daily_usage = pd.read_csv(PROCESSED_DIR / "mart_forecast_features.csv")
fact_report_views = pd.read_csv(PROCESSED_DIR / "fact_report_views.csv")
report_performance = pd.read_csv(PROCESSED_DIR / "mart_report_performance.csv")
dim_report = pd.read_csv(PROCESSED_DIR / "dim_report.csv")
dim_date = pd.read_csv(PROCESSED_DIR / "dim_date.csv")

input_tables = {
    "daily_usage": daily_usage.shape,
    "fact_report_views": fact_report_views.shape,
    "report_performance": report_performance.shape,
    "dim_report": dim_report.shape,
    "dim_date": dim_date.shape,
}
input_tables

{'daily_usage': (13650, 19),
 'fact_report_views': (411006, 6),
 'report_performance': (13627, 7),
 'dim_report': (30, 5),
 'dim_date': (455, 6)}

## 3. Compute Report Features

The feature table contains one row per report. Core features include average daily views, repeat rate, top-user concentration, active days, performance summaries, and recent usage change.

In [3]:
report_features = build_report_features(
    daily_adoption=daily_usage,
    fact_report_views=fact_report_views,
    report_performance=report_performance,
    dim_report=dim_report,
    dim_date=dim_date,
)

report_features_path = METRICS_DIR / "report_features.csv"
report_features.to_csv(report_features_path, index=False)

report_features.head()

,report_id,report_name,avg_views,repeat_rate,top_user_concentration,days_active,total_views,unique_users,avg_load_time,p90_load_time,latest_views,prior_views,usage_change_pct
0,R_001,Report_001,35.076923,1.00,0.017356,455,15960,200,3873.340252,4454.885275,37,39,-0.051282
1,R_002,Report_002,54.142857,1.00,0.016318,455,24635,200,2329.089463,2927.798022,58,52,0.115385
2,R_003,Report_003,20.896703,0.98,0.017564,455,9508,200,3590.541480,4127.471648,31,19,0.631579
3,R_004,Report_004,27.021978,1.00,0.020171,455,12295,200,5318.436678,5881.256044,27,25,0.080000
4,R_005,Report_005,77.914286,1.00,0.013060,455,35451,200,883.997844,1412.701099,79,102,-0.225490


## 4. Review `report_features.csv`

This quick profile checks the generated feature file and gives a compact view of the behavioural metrics.

In [4]:
pd.read_csv(report_features_path).describe(include="all")

,report_id,report_name,avg_views,repeat_rate,top_user_concentration,days_active,total_views,unique_users,avg_load_time,p90_load_time,latest_views,prior_views,usage_change_pct
count,30,30,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
unique,30,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,R_001,Report_001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,46.682857,0.997294,0.016347,454.233333,21240.700000,199.466667,3441.621463,4009.669819,56.233333,53.300000,0.092188
std,NaN,NaN,28.443550,0.006347,0.002793,3.654787,12941.815369,1.591645,1281.415681,1290.475440,34.610825,31.770893,0.303822
min,NaN,NaN,5.384615,0.979695,0.007993,435.000000,2450.000000,192.000000,883.997844,1412.701099,9.000000,6.000000,-0.454545
25%,NaN,NaN,27.721429,1.000000,0.016067,455.000000,12613.250000,200.000000,2541.609479,3141.953297,33.500000,31.000000,-0.093902
50%,NaN,NaN,42.128571,1.000000,0.016796,455.000000,19168.500000,200.000000,3350.061304,3854.266150,50.000000,51.000000,0.085872
75%,NaN,NaN,54.129121,1.000000,0.017697,455.000000,24628.750000,200.000000,4521.201022,5080.092527,78.250000,64.250000,0.152050


## 5. Create Report Segments

Reports are assigned to simple rule-based segments: `high_value`, `niche`, `at_risk`, or `inactive`. Quantiles are used where fixed business thresholds are not available.

In [5]:
report_segments = build_report_segments(report_features)

report_segments_path = SEGMENTS_DIR / "report_segments.csv"
report_segments.to_csv(report_segments_path, index=False)

report_segments.head()

,report_id,report_name,report_segment,segment_reason
0,R_001,Report_001,niche,"Usage is specialized, with healthy repeat beha..."
1,R_002,Report_002,high_value,Average views and unique users are both in the...
2,R_003,Report_003,niche,Usage is moderate or low without a clear healt...
3,R_004,Report_004,niche,"Usage is specialized, with healthy repeat beha..."
4,R_005,Report_005,high_value,Average views and unique users are both in the...


## 6. Review `report_segments.csv`

The segment distribution shows how many reports fall into each business-friendly group.

In [6]:
pd.read_csv(report_segments_path)["report_segment"].value_counts()

report_segment
niche         18
high_value     8
at_risk        4
Name: count, dtype: int64

## 7. Apply Diagnostic Rules

Diagnostics translate the features and segments into health flags for performance, engagement, dependency, and inactive risk.

In [7]:
report_diagnostics = build_report_diagnostics(report_features, report_segments)

report_diagnostics_path = DIAGNOSTICS_DIR / "report_diagnostics.csv"
report_diagnostics.to_csv(report_diagnostics_path, index=False)

report_diagnostics.head()

,report_id,report_name,report_segment,performance_issue,engagement_issue,dependency_risk,inactive_risk,main_diagnostic,diagnostic_summary
0,R_001,Report_001,niche,False,False,False,False,healthy_or_monitor,No major diagnostic rule was triggered; contin...
1,R_002,Report_002,high_value,False,False,False,False,healthy_or_monitor,No major diagnostic rule was triggered; contin...
2,R_003,Report_003,niche,False,False,False,False,healthy_or_monitor,No major diagnostic rule was triggered; contin...
3,R_004,Report_004,niche,False,False,False,False,healthy_or_monitor,No major diagnostic rule was triggered; contin...
4,R_005,Report_005,high_value,False,False,False,False,healthy_or_monitor,No major diagnostic rule was triggered; contin...


## 8. Review `report_diagnostics.csv`

The main diagnostic gives one prioritized health status per report, with a short explanation for business users.

In [8]:
pd.read_csv(report_diagnostics_path)["main_diagnostic"].value_counts()

main_diagnostic
healthy_or_monitor    26
performance_issue      3
engagement_issue       1
Name: count, dtype: int64

## 9. Save Outputs

The three CSV outputs are saved to the expected project folders.

In [9]:
for path in [report_features_path, report_segments_path, report_diagnostics_path]:
    print(path.relative_to(PROJECT_ROOT))

outputs/metrics/report_features.csv
outputs/segments/report_segments.csv
outputs/diagnostics/report_diagnostics.csv


## 10. Brief Next Steps

Useful next steps would be to connect these outputs to the forecasting results, add stakeholder-approved thresholds, and later compare rule-based segments against clustering once the explainable version is accepted.